# Shapiro 时间延迟（T_PM）— C 发射、D 接收、发射点含二阶回推

本 notebook 按你的最新要求重新整理，当前计算的是 **TpMr 的 TPM**，对应过程为：

- **T 星 → C 星（发射端）**
- **M 星 → D 星（接收端）**

也就是统一改为 **C 星发射至 D 星接收**。

## 本版修改要点

1. **发射端/接收端重定义**
   - 原 notebook：D 发射、C 接收
   - 本 notebook：**C 发射、D 接收**

2. **加速度项改为地球点质量引力场**
   \[
   a = \frac{GM}{R^2}
   \]
   其中：
   - \(G\)：万有引力常数
   - \(M\)：地球质量
   - \(R\)：对应卫星到地心的距离

   在程序中为了保持方向信息，实际使用的是向量形式：
   \[
   \mathbf a = -\frac{GM}{R^3}\mathbf r
   \]
   其模长正好等于 \(GM/R^2\)。

3. **发射点回推采用二阶展开**
   在接收时刻 \(t_r\) 评估几何量：
   \[
   d_0=\frac{r_D(t_r)-r_C(t_r)}{|r_D(t_r)-r_C(t_r)|}
   \]
   \[
   \Delta t_{\mathrm{inst}}=\frac{|r_D(t_r)-r_C(t_r)|}{c}
   \]
   \[
   \Delta t_{\mathrm{corr}}=\Delta t_{\mathrm{inst}}\left(1+\frac{d_0\cdot v_C}{c}\right)
   \]
   并采用二阶回推：
   \[
   r_e \approx r_C(t_r)-v_C(t_r)\Delta t_{\mathrm{corr}}+\frac12 a_C(t_r)\Delta t_{\mathrm{corr}}^2
   \]

4. **Shapiro 一程延迟公式**
   \[
   T_{PM}=\frac{2GM}{c^3}\ln\left(\frac{|r_r|+|r_e|+|r_r-r_e|}{|r_r|+|r_e|-|r_r-r_e|}\right)
   \]
   输出等效距离：
   \[
   \rho_{PM}=c\,T_{PM}
   \]

## 输出内容

输出 Excel 将包含：
- `T_PM_s`
- `rho_PM_m`
- `dt_inst_s`
- `dt_corr_s`
- `d0_x, d0_y, d0_z`
- `d0_dot_vC_mps`
- `aC_x_mps2, aC_y_mps2, aC_z_mps2`
- `aC_mag_mps2`

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

# -----------------------------
# 常数
# -----------------------------
C0 = 299792458.0                  # 光速 (m/s)
G = 6.67430e-11                   # 万有引力常数 (m^3 kg^-1 s^-2)
M_EARTH = 5.9722e24               # 地球质量 (kg)
GM_EARTH = G * M_EARTH            # 地球引力常数 (m^3/s^2)

# -----------------------------
# 输入/输出文件
# -----------------------------
GNI_C_PATH = "GNI1B_2022-06-05_C_04.txt"   # C: emitter
GNI_D_PATH = "GNI1B_2022-06-05_D_04.txt"   # D: receiver

OUT_XLSX = "Shapiro_TPM_GNI1B_D_emit_C_recv_d0.xlsx"

In [2]:
def find_first_data_row(filepath: str) -> int:
    """查找 GNI1B 文本中首个数据行。"""
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            if re.match(r"^\s*\d+\s+", line):
                return i
    raise RuntimeError(f"No data rows found in {filepath}")


def read_gni1b(filepath: str, expected_sat: str) -> pd.DataFrame:
    """读取指定卫星的 GNI1B 惯性系位置/速度数据。"""
    cols = [
        "gps_time", "sat_id", "coord_ref",
        "x", "y", "z",
        "xerr", "yerr", "zerr",
        "vx", "vy", "vz",
        "vxerr", "vyerr", "vzerr",
        "qualflg"
    ]

    skip = find_first_data_row(filepath)
    df = pd.read_csv(
        filepath,
        delim_whitespace=True,
        header=None,
        names=cols,
        skiprows=skip
    )

    df = df[(df["sat_id"] == expected_sat) & (df["coord_ref"] == "I")].copy()
    df = df[["gps_time", "x", "y", "z", "vx", "vy", "vz", "qualflg"]]
    df["gps_time"] = df["gps_time"].astype(np.int64)
    return df.sort_values("gps_time").reset_index(drop=True)


def gravitational_acceleration(r_xyz: np.ndarray, gm: float = GM_EARTH) -> np.ndarray:
    """
    地球点质量引力场下的加速度向量：
        a_vec = -(GM / R^3) * r

    其模长满足：
        |a_vec| = GM / R^2
    """
    r_xyz = np.asarray(r_xyz, dtype=float)
    r_norm = np.linalg.norm(r_xyz, axis=-1)

    if np.any(r_norm == 0.0):
        raise ValueError("Satellite position norm is zero; cannot compute gravity acceleration.")

    scale = -gm / (r_norm ** 3)

    if r_xyz.ndim == 1:
        return scale * r_xyz
    return r_xyz * scale[:, None]


def shapiro_delay_oneway(r_e: np.ndarray, r_r: np.ndarray, gm: float = GM_EARTH, c: float = C0) -> float:
    """一程 Shapiro 延迟。"""
    r_e = np.asarray(r_e, dtype=float)
    r_r = np.asarray(r_r, dtype=float)

    re_norm = float(np.linalg.norm(r_e))
    rr_norm = float(np.linalg.norm(r_r))
    R_er = float(np.linalg.norm(r_r - r_e))

    numerator = rr_norm + re_norm + R_er
    denominator = rr_norm + re_norm - R_er

    if denominator <= 0.0:
        return np.nan

    return (2.0 * gm / (c ** 3)) * float(np.log(numerator / denominator))

In [3]:
def solve_emission_point_second_order(
    r_C_tr: np.ndarray,
    v_C_tr: np.ndarray,
    a_C_tr: np.ndarray,
    r_D_tr: np.ndarray,
    c: float = C0
):
    """
    在接收时刻 t_r 评估：
        d0      = (r_D - r_C) / |r_D - r_C|
        dt_inst = |r_D - r_C| / c
        dt_corr = dt_inst * (1 + (d0 · v_C) / c)

    对发射端 C 做二阶回推：
        r_e ≈ r_C(t_r) - v_C(t_r) * dt_corr + 0.5 * a_C(t_r) * dt_corr^2

    返回：
        r_e, dt_inst, dt_corr, d0, d0_dot_vC
    """
    r_C_tr = np.asarray(r_C_tr, dtype=float)
    v_C_tr = np.asarray(v_C_tr, dtype=float)
    a_C_tr = np.asarray(a_C_tr, dtype=float)
    r_D_tr = np.asarray(r_D_tr, dtype=float)

    dr = r_D_tr - r_C_tr
    dist = float(np.linalg.norm(dr))
    if dist == 0.0:
        raise ValueError("r_D_tr and r_C_tr coincide; cannot compute d0.")

    d0 = dr / dist
    dt_inst = dist / c
    d0_dot_vC = float(np.dot(d0, v_C_tr))
    dt_corr = dt_inst * (1.0 + d0_dot_vC / c)

    r_e = r_C_tr - v_C_tr * dt_corr + 0.5 * a_C_tr * (dt_corr ** 2)
    return r_e, dt_inst, dt_corr, d0, d0_dot_vC

In [4]:
# -----------------------------
# 读取并对齐 C / D 轨道数据
# -----------------------------
for fp in [GNI_C_PATH, GNI_D_PATH]:
    if not Path(fp).exists():
        raise FileNotFoundError(f"未找到输入文件：{fp}")

dfC = read_gni1b(GNI_C_PATH, "C")   # emitter
dfD = read_gni1b(GNI_D_PATH, "D")   # receiver

df = (
    dfC.merge(dfD, on="gps_time", suffixes=("_C", "_D"), how="inner")
       .sort_values("gps_time")
       .reset_index(drop=True)
)

# 为发射端 C 计算加速度
rC_all = df[["x_C", "y_C", "z_C"]].to_numpy(dtype=float)
aC_all = gravitational_acceleration(rC_all)

df["ax_C"] = aC_all[:, 0]
df["ay_C"] = aC_all[:, 1]
df["az_C"] = aC_all[:, 2]

print("rows:", len(df))
df.head()

rows: 86400


,gps_time,x_C,y_C,z_C,vx_C,vy_C,vz_C,qualflg_C,x_D,y_D,z_D,vx_D,vy_D,vz_D,qualflg_D,ax_C,ay_C,az_C
0,707659200,3.716399e+06,3.208044e+06,4.780815e+06,-4149.229192,-3352.914250,5461.061418,10000000,3.820699e+06,3.292583e+06,4.639412e+06,-4030.053419,-3250.486988,5610.213439,10000000,-4.603404,-3.973719,-5.921868
1,707659201,3.712248e+06,3.204689e+06,4.786273e+06,-4153.820680,-3356.877897,5455.131498,10000000,3.816666e+06,3.289331e+06,4.645020e+06,-4034.774291,-3254.555538,5604.458237,10000000,-4.598282,-3.969580,-5.928654
2,707659202,3.708092e+06,3.201331e+06,4.791725e+06,-4158.407018,-3360.837382,5449.194835,10000000,3.812629e+06,3.286074e+06,4.650621e+06,-4039.490160,-3258.620051,5598.696106,10000000,-4.593154,-3.965437,-5.935433
3,707659203,3.703931e+06,3.197968e+06,4.797171e+06,-4162.988200,-3364.792697,5443.251438,10000000,3.808587e+06,3.282814e+06,4.656217e+06,-4044.201020,-3262.680523,5592.927053,10000000,-4.588020,-3.961289,-5.942205
4,707659204,3.699766e+06,3.194601e+06,4.802612e+06,-4167.564221,-3368.743840,5437.301313,10000000,3.804541e+06,3.279549e+06,4.661807e+06,-4048.906867,-3266.736949,5587.151084,10000000,-4.582880,-3.957135,-5.948970


In [5]:
# -----------------------------
# 批量计算 T_PM 和 rho_PM
# -----------------------------
rows = []

for row in df.itertuples(index=False):
    gps_time = int(row.gps_time)

    # C: emitter
    rC_tr = np.array([row.x_C, row.y_C, row.z_C], dtype=float)
    vC_tr = np.array([row.vx_C, row.vy_C, row.vz_C], dtype=float)
    aC_tr = np.array([row.ax_C, row.ay_C, row.az_C], dtype=float)

    # D: receiver
    rD_tr = np.array([row.x_D, row.y_D, row.z_D], dtype=float)

    # 二阶回推发射点
    r_e, dt_inst, dt_corr, d0, d0_dot_vC = solve_emission_point_second_order(
        r_C_tr=rC_tr,
        v_C_tr=vC_tr,
        a_C_tr=aC_tr,
        r_D_tr=rD_tr,
        c=C0
    )

    # 接收点
    r_r = rD_tr

    # 一程 Shapiro 延迟
    T_PM = shapiro_delay_oneway(r_e=r_e, r_r=r_r, gm=GM_EARTH, c=C0)
    rho_PM = C0 * T_PM

    rows.append((
        gps_time,
        T_PM,
        rho_PM,
        dt_inst,
        dt_corr,
        d0[0], d0[1], d0[2],
        d0_dot_vC,
        aC_tr[0], aC_tr[1], aC_tr[2],
        float(np.linalg.norm(aC_tr)),
        r_e[0], r_e[1], r_e[2]
    ))

df_out = pd.DataFrame(
    rows,
    columns=[
        "gps_time",
        "T_PM_s",
        "rho_PM_m",
        "dt_inst_s",
        "dt_corr_s",
        "d0_x", "d0_y", "d0_z",
        "d0_dot_vC_mps",
        "aC_x_mps2", "aC_y_mps2", "aC_z_mps2",
        "aC_mag_mps2",
        "r_e_x_m", "r_e_y_m", "r_e_z_m"
    ]
)

df_out.to_excel(OUT_XLSX, index=False)
df_out.head()

,gps_time,T_PM_s,rho_PM_m,dt_inst_s,dt_corr_s,d0_x,d0_y,d0_z,d0_dot_vC_mps,aC_x_mps2,aC_y_mps2,aC_z_mps2,aC_mag_mps2,r_e_x_m,r_e_y_m,r_e_z_m
0,707659200,8.419040e-13,0.000252,0.00065,0.00065,0.534905,0.433562,-0.725190,-7633.447095,-4.603404,-3.973719,-5.921868,8.488244,3.716402e+06,3.208046e+06,4.780811e+06
1,707659201,8.419052e-13,0.000252,0.00065,0.00065,0.535515,0.434087,-0.724425,-7633.446221,-4.598282,-3.969580,-5.928654,8.488269,3.712251e+06,3.204692e+06,4.786270e+06
2,707659202,8.419064e-13,0.000252,0.00065,0.00065,0.536126,0.434612,-0.723659,-7633.445323,-4.593154,-3.965437,-5.935433,8.488293,3.708094e+06,3.201333e+06,4.791722e+06
3,707659203,8.419076e-13,0.000252,0.00065,0.00065,0.536735,0.435136,-0.722892,-7633.444399,-4.588020,-3.961289,-5.942205,8.488318,3.703934e+06,3.197970e+06,4.797168e+06
4,707659204,8.419088e-13,0.000252,0.00065,0.00065,0.537344,0.435659,-0.722124,-7633.443450,-4.582880,-3.957135,-5.948970,8.488342,3.699768e+06,3.194603e+06,4.802608e+06


In [ ]:
print("rows =", len(df_out))
print("T_PM (s):    min / mean / max =", df_out["T_PM_s"].min(), df_out["T_PM_s"].mean(), df_out["T_PM_s"].max())
print("rho_PM (m):  min / mean / max =", df_out["rho_PM_m"].min(), df_out["rho_PM_m"].mean(), df_out["rho_PM_m"].max())
print("rho_PM (um): min / mean / max =", 1e6 * df_out["rho_PM_m"].min(), 1e6 * df_out["rho_PM_m"].mean(), 1e6 * df_out["rho_PM_m"].max())
print("dt_inst (s): min / mean / max =", df_out["dt_inst_s"].min(), df_out["dt_inst_s"].mean(), df_out["dt_inst_s"].max())
print("dt_corr (s): min / mean / max =", df_out["dt_corr_s"].min(), df_out["dt_corr_s"].mean(), df_out["dt_corr_s"].max())
print("saved to:", OUT_XLSX)